## Cell 1: Install Dependencies and Download JARs

This cell installs the required Python packages: a compatible version of NumPy, PySpark, Delta Spark, pandas-datareader, and PyArrow. It also manually downloads two JAR files — the Hadoop AWS connector and the AWS Java SDK bundle — into a local `/content/jars` directory. These JARs are required for Spark to communicate with Amazon S3 using the S3A filesystem protocol. Finally, it prints the size of each downloaded JAR to confirm they were fetched correctly.

In [ ]:
!pip install -q "numpy<2.0"
!pip install -q pyspark==3.5.3 delta-spark==3.2.1
!pip install -q pandas-datareader "pyarrow==14.0.1"

# Download JARs manually to avoid network issues during Spark session
import os
os.makedirs("/content/jars", exist_ok=True)

!wget -q "https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar" \
     -O /content/jars/hadoop-aws.jar

!wget -q "https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar" \
     -O /content/jars/aws-sdk.jar

# Verify sizes
for f in os.listdir("/content/jars"):
    mb = os.path.getsize(f"/content/jars/{f}") // (1024*1024)
    print(f" {f} → {mb} MB")

 delta-core.jar → 4 MB
 delta-storage.jar → 0 MB
 aws-sdk.jar → 267 MB
 hadoop-aws.jar → 0 MB


## Cell 2: Load AWS Credentials from Colab Secrets

This cell reads AWS credentials and the S3 bucket name from Google Colab's secret storage and sets them as environment variables. This approach keeps sensitive keys out of the notebook source code. The variables set here — `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`, and `S3_BUCKET_NAME` — are referenced throughout the rest of the pipeline.

In [ ]:
import os
from google.colab import userdata

os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = userdata.get("AWS_DEFAULT_REGION")
os.environ["S3_BUCKET_NAME"]        = userdata.get("S3_BUCKET_NAME")
print(" Secrets loaded!")

 Secrets loaded!


## Cell 3: Initialize Spark Session with Delta Lake and S3 Support

This cell creates a local PySpark session configured with two key extensions: Delta Lake support (for ACID-compliant table writes) and S3A filesystem support (for reading and writing directly to Amazon S3). The locally downloaded JARs from Cell 1 are loaded here instead of fetching them from Maven at runtime, which avoids network issues. After startup, the cell runs a quick smoke test — writing a small Delta table locally and then writing to S3 — to confirm both integrations are working before any real data is processed.

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import os

# Use manually downloaded JARs — bypass spark.jars.packages entirely
builder = (
    SparkSession.builder
    .appName("StockPipeline")
    .master("local[2]")

    # Load JARs from local filesystem
    .config("spark.jars",
            "/content/jars/hadoop-aws.jar,"
            "/content/jars/aws-sdk.jar")

    # Delta
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")

    # S3A
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.access.key",
            os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.hadoop.fs.s3a.secret.key",
            os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.endpoint",
            "s3.amazonaws.com")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")

    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "4g")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f" Spark {spark.version} ready!")

# Test Delta locally
spark.range(5).write.format("delta").mode("overwrite").save("/tmp/test_delta")
print(" Delta working!")

# Test S3
bucket = os.environ["S3_BUCKET_NAME"]
try:
    spark.range(3).write.format("delta").mode("overwrite").save(
        f"s3a://{bucket}/test/spark_test"
    )
    print(" S3 write working!")
except Exception as e:
    print(f" S3 failed: {e}")

 Spark 3.5.3 ready!
 Delta working!
 S3 write working!


## Cell 4: Bronze Layer — Raw Stock Data Ingestion

This cell implements the Bronze (raw) layer of a Medallion architecture data pipeline. It fetches daily OHLCV (Open, High, Low, Close, Volume) stock data for five tickers — AAPL, MSFT, GOOGL, AMZN, and TSLA — from Stooq via `pandas_datareader`, covering the period from 2020-01-01 to 2024-12-31. The data is enriched with metadata columns (ticker, ingestion date, ingestion timestamp, source), converted to a Spark DataFrame with an explicit schema, and written to Amazon S3 as a partitioned Delta Lake table. A read-back verification confirms row counts, column count, schema, and sample data after the write.

In [ ]:
import pandas_datareader as pdr
import pandas as pd
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, LongType, DateType
)
from datetime import datetime, date
import time

BUCKET         = os.environ["S3_BUCKET_NAME"]
BRONZE_PATH    = f"s3a://{BUCKET}/bronze/stocks"
TICKERS        = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]
START_DATE     = "2020-01-01"
END_DATE       = "2024-12-31"
INGESTION_DATE = date.today().strftime("%Y-%m-%d")

print("=" * 50)
print("     BRONZE LAYER — RAW DATA INGESTION")
print("=" * 50)

# ── Fetch ──────────────────────────────────────────
all_data = []
for ticker in TICKERS:
    print(f" Fetching {ticker}...")
    try:
        df = pdr.get_data_stooq(ticker, start=START_DATE, end=END_DATE)
        df = df.reset_index()
        df.columns = [c.lower() for c in df.columns]
        df["ticker"]         = ticker
        df["ingestion_date"] = INGESTION_DATE
        df["ingestion_ts"]   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        df["source"]         = "stooq"
        df = df.sort_values("date").reset_index(drop=True)
        all_data.append(df)
        print(f"    {len(df):,} rows")
        time.sleep(0.5)
    except Exception as e:
        print(f"    {e}")

pandas_df = pd.concat(all_data, ignore_index=True)
print(f"\n Total : {len(pandas_df):,} rows")

# ── Fix types ──────────────────────────────────────
pandas_df["date"]   = pd.to_datetime(pandas_df["date"]).dt.date
pandas_df["open"]   = pandas_df["open"].astype(float)
pandas_df["high"]   = pandas_df["high"].astype(float)
pandas_df["low"]    = pandas_df["low"].astype(float)
pandas_df["close"]  = pandas_df["close"].astype(float)
pandas_df["volume"] = pandas_df["volume"].fillna(0).astype("int64")

pandas_df = pandas_df[[
    "ticker", "date", "open", "high", "low",
    "close", "volume", "ingestion_date", "ingestion_ts", "source"
]]

# ── Schema ─────────────────────────────────────────
schema = StructType([
    StructField("ticker",         StringType(),  nullable=False),
    StructField("date",           DateType(),    nullable=False),
    StructField("open",           DoubleType(),  nullable=True),
    StructField("high",           DoubleType(),  nullable=True),
    StructField("low",            DoubleType(),  nullable=True),
    StructField("close",          DoubleType(),  nullable=True),
    StructField("volume",         LongType(),    nullable=True),
    StructField("ingestion_date", StringType(),  nullable=False),
    StructField("ingestion_ts",   StringType(),  nullable=False),
    StructField("source",         StringType(),  nullable=False),
])

# ── Convert to Spark ───────────────────────────────
print("\n Converting to Spark...")
spark_df = spark.createDataFrame(pandas_df, schema=schema)
print(f" Spark DF ready! | Partitions: {spark_df.rdd.getNumPartitions()}")

# ── Write Delta to S3 ──────────────────────────────
output_path = f"{BRONZE_PATH}/ingestion_date={INGESTION_DATE}"
print(f"\n Writing Delta to S3...")
print(f"   {output_path}")
spark_df.write.format("delta").mode("overwrite").save(output_path)
print(" Write complete!")

# ── Verify ─────────────────────────────────────────
print("\n Verifying from S3...")
verify_df = spark.read.format("delta").load(output_path)
print(f" Rows confirmed  : {verify_df.count():,}")
print(f" Columns         : {len(verify_df.columns)}")
print("\n Schema:")
verify_df.printSchema()
print("\n Sample rows:")
verify_df.show(5, truncate=False)
print("\n Rows per ticker:")
verify_df.groupBy("ticker").count().orderBy("ticker").show()

print("\n" + "=" * 50)
print("    BRONZE LAYER COMPLETE!")
print(f"    {output_path}")
print(f"    Format : Delta Lake")
print("=" * 50)

     BRONZE LAYER — RAW DATA INGESTION
 Fetching AAPL...
    1,258 rows
 Fetching MSFT...
    1,258 rows
 Fetching GOOGL...
    1,258 rows
 Fetching AMZN...
    1,258 rows
 Fetching TSLA...
    1,258 rows

 Total : 6,290 rows

 Converting to Spark...
 Spark DF ready! | Partitions: 2

 Writing Delta to S3...
   s3a://stock-data-lake-09/bronze/stocks/ingestion_date=2026-02-21
 Write complete!

 Verifying from S3...
 Rows confirmed  : 6,290
 Columns         : 10

 Schema:
root
 |-- ticker: string (nullable = true)
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- ingestion_date: string (nullable = true)
 |-- ingestion_ts: string (nullable = true)
 |-- source: string (nullable = true)


 Sample rows:
+------+----------+-------+-------+-------+-------+--------+--------------+-------------------+------+
|ticker|date      |ope

## Cell 5: Bronze Path Confirmation

This is a quick sanity check cell that prints the S3 path where the Bronze layer data was written. It serves as a visual confirmation before proceeding to the Silver layer transformation step.

In [ ]:
# Quick confirmation before Silver Layer
print("Bronze path ready:", f"s3a://{os.environ['S3_BUCKET_NAME']}/bronze/stocks/ingestion_date=2026-02-21")
print("Moving to Silver Layer...")

Bronze path ready: s3a://stock-data-lake-09/bronze/stocks/ingestion_date=2026-02-21
Moving to Silver Layer...


## Cell 6: Silver Layer — Cleaning, Validation, and Data Quality Checks

This cell reads the raw Bronze Delta table and applies a series of data quality transformations before writing the cleaned result to the Silver layer. The steps include: (1) deduplication on ticker and date to eliminate any duplicate rows from reprocessing; (2) null handling — dropping rows with missing close prices and filling missing OHLC values with the close price; (3) type casting — ensuring the date column is a proper DateType and rounding price columns to four decimal places; (4) a structured data quality report checking for negative prices, invalid high/low relationships, presence of all five tickers, future dates, and negative volume values. The final cleaned DataFrame is written to S3 as a Delta table and verified with a read-back.

In [ ]:
# CELL 5 — SILVER LAYER
# Read Bronze → Clean → Validate → Write to Silver

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, LongType, DateType
)

BUCKET       = os.environ["S3_BUCKET_NAME"]
BRONZE_PATH  = f"s3a://{BUCKET}/bronze/stocks/ingestion_date=2026-02-21"
SILVER_PATH  = f"s3a://{BUCKET}/silver/stocks"
SILVER_OUT   = f"{SILVER_PATH}/ingestion_date=2026-02-21"

print("=" * 50)
print("     SILVER LAYER — CLEANING & VALIDATION")
print("=" * 50)

# ── Step 1: Read Bronze ────────────────────────────
print("\n Reading Bronze layer...")
bronze_df = spark.read.format("delta").load(BRONZE_PATH)
bronze_count = bronze_df.count()
print(f" Rows read from Bronze : {bronze_count:,}")

# ── Step 2: Deduplicate ────────────────────────────
# Why? Same ticker+date should never appear twice.
# If pipeline runs twice, Bronze gets duplicates.
# Silver must be deduplicated — one row per ticker per date.
print("\n Deduplicating...")
dedup_df = bronze_df.dropDuplicates(["ticker", "date"])
dedup_count = dedup_df.count()
print(f" Rows after dedup      : {dedup_count:,}")
print(f"   Duplicates removed   : {bronze_count - dedup_count:,}")

# ── Step 3: Handle Nulls ───────────────────────────
# Why? Null prices would break SMA/return calculations in Gold.
# Strategy:
#   - Drop rows where close is null (can't calculate returns without it)
#   - Fill open/high/low nulls with close price (reasonable approximation)
#   - Fill volume nulls with 0
print("\n Handling nulls...")

# Check nulls before
print("   Null counts before cleaning:")
bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["open", "high", "low", "close", "volume"]
]).show()

cleaned_df = (
    dedup_df
    # Drop rows with null close — these are unusable
    .filter(F.col("close").isNotNull())
    .filter(F.col("date").isNotNull())
    .filter(F.col("ticker").isNotNull())
    # Fill missing OHLC with close price
    .withColumn("open",   F.coalesce(F.col("open"),   F.col("close")))
    .withColumn("high",   F.coalesce(F.col("high"),   F.col("close")))
    .withColumn("low",    F.coalesce(F.col("low"),    F.col("close")))
    # Fill missing volume with 0
    .withColumn("volume", F.coalesce(F.col("volume"), F.lit(0)))
)

print(f" Rows after null clean  : {cleaned_df.count():,}")

# ── Step 4: Data Type Casting & Formatting ─────────
# Why? Bronze stores ingestion_date as string.
# Silver enforces proper types for all columns.
print("\n Casting types...")

typed_df = (
    cleaned_df
    # Ensure date is proper DateType
    .withColumn("date", F.col("date").cast(DateType()))
    # Round prices to 4 decimal places
    .withColumn("open",  F.round(F.col("open"),  4))
    .withColumn("high",  F.round(F.col("high"),  4))
    .withColumn("low",   F.round(F.col("low"),   4))
    .withColumn("close", F.round(F.col("close"), 4))
    # Cast ingestion_date string to proper date
    .withColumn("ingestion_date",
                F.to_date(F.col("ingestion_date"), "yyyy-MM-dd"))
    # Add silver processing timestamp
    .withColumn("silver_ts",
                F.lit(F.current_timestamp()).cast("string"))
)

print(" Types cast successfully!")

# ── Step 5: Data Quality Checks ───────────────────
# Why? Catch bad data before it reaches Gold layer.
# These checks mirror what you'd find in production
# data quality frameworks like Great Expectations.
print("\n Running data quality checks...")

total      = typed_df.count()
dq_results = {}

# Check 1: No negative prices
neg_prices = typed_df.filter(
    (F.col("close") < 0) |
    (F.col("open")  < 0) |
    (F.col("high")  < 0) |
    (F.col("low")   < 0)
).count()
dq_results["No negative prices"] = neg_prices == 0

# Check 2: High >= Low always
invalid_hl = typed_df.filter(
    F.col("high") < F.col("low")
).count()
dq_results["High >= Low"] = invalid_hl == 0

# Check 3: All 5 tickers present
ticker_count = typed_df.select("ticker").distinct().count()
dq_results["All 5 tickers present"] = ticker_count == 5

# Check 4: No future dates
future_dates = typed_df.filter(
    F.col("date") > F.current_date()
).count()
dq_results["No future dates"] = future_dates == 0

# Check 5: Volume non-negative
neg_volume = typed_df.filter(F.col("volume") < 0).count()
dq_results["No negative volume"] = neg_volume == 0

# Print DQ results
print("\n   Data Quality Report:")
print("   " + "─" * 35)
all_passed = True
for check, passed in dq_results.items():
    status = " PASS" if passed else " FAIL"
    print(f"   {status} | {check}")
    if not passed:
        all_passed = False

print("   " + "─" * 35)
print(f"   Overall: {' ALL CHECKS PASSED' if all_passed else ' SOME CHECKS FAILED'}")

# ── Step 6: Select Final Silver Columns ───────────
print("\n Selecting final Silver schema...")

silver_df = typed_df.select(
    "ticker",
    "date",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "ingestion_date",
    "silver_ts",
    "source"
).orderBy("ticker", "date")

# ── Step 7: Write to S3 as Delta ──────────────────
print(f"\n Writing Silver to S3...")
print(f"   {SILVER_OUT}")

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(SILVER_OUT)

print(" Silver write complete!")

# ── Step 8: Verify ─────────────────────────────────
print("\n Verifying Silver layer...")
verify_df = spark.read.format("delta").load(SILVER_OUT)

print(f" Rows confirmed  : {verify_df.count():,}")
print(f" Columns         : {len(verify_df.columns)}")
print("\n Silver Schema:")
verify_df.printSchema()
print("\n Sample rows:")
verify_df.show(5, truncate=False)
print("\n Rows per ticker:")
verify_df.groupBy("ticker").count().orderBy("ticker").show()
print("\n Date range per ticker:")
verify_df.groupBy("ticker").agg(
    F.min("date").alias("first_date"),
    F.max("date").alias("last_date")
).orderBy("ticker").show()

print("\n" + "=" * 50)
print("    SILVER LAYER COMPLETE!")
print(f"    {SILVER_OUT}")
print(f"    Format  : Delta Lake")
print(f"    Cleaned : Deduped + Nulls + Types + DQ checks")
print("=" * 50)

     SILVER LAYER — CLEANING & VALIDATION

 Reading Bronze layer...
 Rows read from Bronze : 6,290

 Deduplicating...
 Rows after dedup      : 6,290
   Duplicates removed   : 0

 Handling nulls...
   Null counts before cleaning:
+----+----+---+-----+------+
|open|high|low|close|volume|
+----+----+---+-----+------+
|   0|   0|  0|    0|     0|
+----+----+---+-----+------+

 Rows after null clean  : 6,290

 Casting types...
 Types cast successfully!

 Running data quality checks...

   Data Quality Report:
   ───────────────────────────────────
    PASS | No negative prices
    PASS | High >= Low
    PASS | All 5 tickers present
    PASS | No future dates
    PASS | No negative volume
   ───────────────────────────────────
   Overall:  ALL CHECKS PASSED

 Selecting final Silver schema...

 Writing Silver to S3...
   s3a://stock-data-lake-09/silver/stocks/ingestion_date=2026-02-21
 Silver write complete!

 Verifying Silver layer...
 Rows confirmed  : 6,290
 Columns         : 10

 Silver S

## Cell 7: Gold Layer — Financial Metrics Computation

This cell reads the cleaned Silver layer data and computes five financial metrics using Spark window functions, then writes the enriched result to the Gold layer. The metrics computed are: (1) `daily_return` — the percentage price change from the previous day's close, calculated using a lag window; (2) `sma_50` — the 50-day simple moving average of closing prices, indicating short-term trend; (3) `sma_200` — the 200-day simple moving average, representing the long-term trend; (4) `volatility_30d` — the 30-day rolling standard deviation of closing prices, used as a measure of risk; and (5) `golden_cross` — a categorical signal set to 'bullish' when SMA-50 exceeds SMA-200, 'bearish' when it falls below, and 'neutral' otherwise. The Gold table is partitioned by year and month for efficient querying, then verified with sample output and a volatility ranking.

In [ ]:
# CELL 6 — GOLD LAYER
# Read Silver → Calculate Financial Metrics → Write Gold

from pyspark.sql import functions as F
from pyspark.sql.window import Window

BUCKET      = os.environ["S3_BUCKET_NAME"]
SILVER_PATH = f"s3a://{BUCKET}/silver/stocks/ingestion_date=2026-02-21"
GOLD_PATH   = f"s3a://{BUCKET}/gold/stock_metrics"

print("=" * 50)
print("     GOLD LAYER — FINANCIAL METRICS")
print("=" * 50)

# ── Step 1: Read Silver ────────────────────────────
print("\n Reading Silver layer...")
silver_df = spark.read.format("delta").load(SILVER_PATH)
print(f" Rows read : {silver_df.count():,}")

# ── Step 2: Define Window Specs ───────────────────
# What is a Window function?
# It performs calculations ACROSS rows related to current row
# WITHOUT collapsing them into groups like GROUP BY does.
#
# Example: For AAPL on 2024-01-15, SMA-50 looks at the
# previous 50 rows for AAPL only, calculates average close.
# The row itself stays — we just ADD a new column.
#
# partitionBy("ticker") = calculate separately per stock
# orderBy("date")       = look at rows in date order
# rowsBetween(-49, 0)   = current row + 49 rows before it = 50 rows

# Window for 50-day calculations
w50 = (
    Window
    .partitionBy("ticker")
    .orderBy("date")
    .rowsBetween(-49, 0)   # 50 rows: current + 49 before
)

# Window for 200-day calculations
w200 = (
    Window
    .partitionBy("ticker")
    .orderBy("date")
    .rowsBetween(-199, 0)  # 200 rows: current + 199 before
)

# Window for daily return (needs previous row only)
w_ret = (
    Window
    .partitionBy("ticker")
    .orderBy("date")
)

# Window for 30-day rolling volatility
w30 = (
    Window
    .partitionBy("ticker")
    .orderBy("date")
    .rowsBetween(-29, 0)   # 30 rows
)

print(" Window specs defined!")

# ── Step 3: Calculate All Metrics ─────────────────
print("\n Calculating financial metrics...")

gold_df = (
    silver_df

    # ── Daily Return ──────────────────────────────
    # Formula: (today's close - yesterday's close) / yesterday's close
    # Why useful? Shows % gain/loss each day.
    # lag(1) = previous row's value within same ticker partition
    .withColumn(
        "daily_return",
        F.round(
            (F.col("close") - F.lag("close", 1).over(w_ret)) /
            F.lag("close", 1).over(w_ret) * 100,
            4
        )
    )

    # ── SMA-50 (50-day Simple Moving Average) ─────
    # Formula: average of last 50 closing prices
    # Why useful? Shows short-term price trend.
    # Traders buy when price crosses above SMA-50.
    .withColumn(
        "sma_50",
        F.round(F.avg("close").over(w50), 4)
    )

    # ── SMA-200 (200-day Simple Moving Average) ───
    # Formula: average of last 200 closing prices
    # Why useful? Shows long-term price trend.
    # "Golden Cross" = SMA-50 crosses above SMA-200 (bullish signal)
    .withColumn(
        "sma_200",
        F.round(F.avg("close").over(w200), 4)
    )

    # ── Rolling Volatility (30-day) ───────────────
    # Formula: standard deviation of daily returns over 30 days
    # Why useful? Measures risk. High volatility = risky stock.
    # TSLA typically has higher volatility than MSFT.
    .withColumn(
        "volatility_30d",
        F.round(F.stddev("close").over(w30), 4)
    )

    # ── Golden Cross Signal ───────────────────────
    # When SMA-50 > SMA-200 = bullish (upward trend)
    # When SMA-50 < SMA-200 = bearish (downward trend)
    # This is one of the most famous trading signals
    .withColumn(
        "golden_cross",
        F.when(F.col("sma_50") > F.col("sma_200"), "bullish")
         .when(F.col("sma_50") < F.col("sma_200"), "bearish")
         .otherwise("neutral")
    )

    # ── Add year/month for partitioning ──────────
    .withColumn("year",  F.year("date"))
    .withColumn("month", F.month("date"))

    # ── Add gold processing timestamp ─────────────
    .withColumn("gold_ts",
                F.lit(F.current_timestamp()).cast("string"))
)

print(" All metrics calculated!")

# ── Step 4: Select Final Gold Columns ─────────────
gold_final = gold_df.select(
    "ticker",
    "date",
    "year",
    "month",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "daily_return",
    "sma_50",
    "sma_200",
    "volatility_30d",
    "golden_cross",
    "gold_ts"
).orderBy("ticker", "date")

# ── Step 5: Write to S3 as Delta ──────────────────
# Partition by year/month for faster Athena queries
# When you query "give me 2024 data", Spark skips all other years
print(f"\n Writing Gold to S3...")
print(f"   {GOLD_PATH}")

gold_final.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .save(GOLD_PATH)

print(" Gold write complete!")

# ── Step 6: Verify ─────────────────────────────────
print("\n Verifying Gold layer...")
verify_df = spark.read.format("delta").load(GOLD_PATH)

print(f" Rows confirmed  : {verify_df.count():,}")
print(f" Columns         : {len(verify_df.columns)}")

print("\n Gold Schema:")
verify_df.printSchema()

print("\n Sample rows (AAPL first 5):")
verify_df.filter(F.col("ticker") == "AAPL") \
         .orderBy("date") \
         .show(5, truncate=False)

print("\n SMA signals — latest date per ticker:")
verify_df.groupBy("ticker").agg(
    F.max("date").alias("latest_date")
).join(verify_df, ["ticker"]) \
 .filter(F.col("date") == F.col("latest_date")) \
 .select("ticker", "date", "close", "sma_50",
         "sma_200", "golden_cross", "daily_return") \
 .orderBy("ticker") \
 .show(truncate=False)

print("\n Average volatility per ticker (riskiest stocks):")
verify_df.groupBy("ticker").agg(
    F.round(F.avg("volatility_30d"), 2).alias("avg_volatility"),
    F.round(F.avg("daily_return"),   4).alias("avg_daily_return")
).orderBy(F.desc("avg_volatility")).show()

print("\n" + "=" * 50)
print("    GOLD LAYER COMPLETE!")
print(f"    {GOLD_PATH}")
print(f"    Metrics : daily_return, sma_50, sma_200,")
print(f"               volatility_30d, golden_cross")
print(f"   ️  Partitioned by year/month")
print("=" * 50)

     GOLD LAYER — FINANCIAL METRICS

 Reading Silver layer...
 Rows read : 6,290
 Window specs defined!

 Calculating financial metrics...
 All metrics calculated!

 Writing Gold to S3...
   s3a://stock-data-lake-09/gold/stock_metrics
 Gold write complete!

 Verifying Gold layer...
 Rows confirmed  : 6,290
 Columns         : 15

 Gold Schema:
root
 |-- ticker: string (nullable = true)
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- daily_return: double (nullable = true)
 |-- sma_50: double (nullable = true)
 |-- sma_200: double (nullable = true)
 |-- volatility_30d: double (nullable = true)
 |-- golden_cross: string (nullable = true)
 |-- gold_ts: string (nullable = true)


 Sample rows (AAPL first 5):
+------+----------+----+-----+-------+----

## Cell 8: Read and Print Gold Layer Schema

This cell reads the Gold Delta table back from S3 and prints its full schema. It serves as a final confirmation that the Gold layer is accessible and structured correctly, showing all 15 columns including the computed financial metrics.

In [ ]:
# Read Gold from S3 and print schema
gold_df = spark.read.format("delta").load(
    f"s3a://{os.environ['S3_BUCKET_NAME']}/gold/stock_metrics"
)
gold_df.printSchema()

root
 |-- ticker: string (nullable = true)
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- daily_return: double (nullable = true)
 |-- sma_50: double (nullable = true)
 |-- sma_200: double (nullable = true)
 |-- volatility_30d: double (nullable = true)
 |-- golden_cross: string (nullable = true)
 |-- gold_ts: string (nullable = true)

